In [55]:
import pandas as pd
import cv2

validation_labels = "hw1_dataset/labels_validation.csv"
test_labels = "hw1_dataset/labels_test.csv"

# test + train için 
numberOfCarImages = 400
numberOfNonCarImages = numberOfCarImages * 3
printDebug = True
displayImages = True
randomSeed = 42

train_labels1 = pd.read_csv("kaggle_dataset/labels_train.csv")
train_labels2 = pd.read_csv("kaggle_dataset/labels_trainval.csv")
train_labels3 = pd.read_csv("kaggle_dataset/labels_val.csv")

train_labels = pd.concat([train_labels1, train_labels2, train_labels3], ignore_index=True)
# duplicate girdileri at
train_labels = train_labels.drop_duplicates()

# class_id = 2 sil. Kamyon 
frame_names_with_truck = train_labels[train_labels['class_id'] == 2]['frame']
train_labels = train_labels[~train_labels['frame'].isin(frame_names_with_truck)]

if(printDebug):
    print(train_labels)
    print(train_labels.shape)
    #class_id = 2 sıfır olmalı 
    print(train_labels[train_labels['class_id'] == 2])


                          frame  xmin  xmax  ymin  ymax  class_id
0       1478019952686311006.jpg   237   251   143   155         1
1       1478019952686311006.jpg   437   454   120   186         3
2       1478019953180167674.jpg   218   231   146   158         1
51      1478019959681353555.jpg     2   135   130   230         1
52      1478019959681353555.jpg   166   178   144   152         1
...                         ...   ...   ...   ...   ...       ...
227572  1479498507972317218.jpg   300   311   135   144         1
227573  1479498507972317218.jpg   324   362   135   147         1
227574  1479498507972317218.jpg   401   431   127   143         1
227575  1479498507972317218.jpg   425   452   129   139         1
227576  1479498507972317218.jpg   440   473   129   143         1

[117712 rows x 6 columns]
(117712, 6)
Empty DataFrame
Columns: [frame, xmin, xmax, ymin, ymax, class_id]
Index: []


In [56]:
# Sadece bir araba içeren frameleri bul

# araba dışındakileri sil
from more_itertools import first


frames_only_with_car = train_labels.loc[train_labels['class_id'] == 1]

# araba resmin en az %20 sinde olsun.
def calculate_image_ratio(row):
    image = cv2.imread("kaggle_dataset/images/"+row["frame"])
    h, w = image.shape[:2]
    image_area = h*w
    car_area = (row['xmax'] - row['xmin']) * (row['ymax'] - row['ymin'])
    return car_area / image_area

frames_only_with_car["car_ratio"] = frames_only_with_car.apply(calculate_image_ratio, axis=1)
frames_only_with_car = frames_only_with_car[frames_only_with_car["car_ratio"] >= 0.20]
frames_only_with_car.drop(columns=["car_ratio"], inplace = True)


# Birden fazla araba olanları sil
frames_only_with_car = frames_only_with_car.drop_duplicates(subset=['frame'], keep=False)

# seç
n = min(numberOfCarImages, frames_only_with_car.shape[0])
frames_only_with_car = frames_only_with_car.sample(n=n, random_state=randomSeed)
numberOfCarImages = n

# contains_car adında bool bir sütun ekle
frames_only_with_car['contains_car'] = True

# "frame" girdisini tam dizin yolu olarak değiştir. kaggle_dataset/images/frame olacak
frames_only_with_car["frame"] = (
    "kaggle_dataset/images/" +
    frames_only_with_car["frame"].astype(str)
)


if(displayImages):
    for index, row in frames_only_with_car.iterrows():
        image_path = row['frame']
        image = cv2.imread(image_path)
        cv2.rectangle(image, (row['xmin'], row['ymin']), (row['xmax'], row['ymax']), (0, 255, 0), 2)
        cv2.imshow("Only Car Frames", image)
        print(row['frame'])
        key = cv2.waitKey(0)
        if key == 27:  # ESC tuşuna basıldığında döngüyü kır
            break
    cv2.destroyAllWindows()

if(printDebug):
    print(frames_only_with_car)
    print(frames_only_with_car.shape)

C:\Users\alibe\AppData\Local\Temp\ipykernel_20748\4225225219.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  frames_only_with_car["car_ratio"] = frames_only_with_car.apply(calculate_image_ratio, axis=1)


kaggle_dataset/images/1479504982411882087.jpg
                                                frame  xmin  xmax  ymin  ymax  \
118040  kaggle_dataset/images/1479504982411882087.jpg   167   353   107   267   
91594   kaggle_dataset/images/1479502451741442800.jpg   148   349    92   262   
35483   kaggle_dataset/images/1478896736412277064.jpg    63   336   115   286   
95192   kaggle_dataset/images/1479502742261490473.jpg   123   309   105   267   
220697  kaggle_dataset/images/1478901218751333990.jpg     0   247    90   285   
...                                               ...   ...   ...   ...   ...   
91690   kaggle_dataset/images/1479502463222077737.jpg   132   355    83   270   
38513   kaggle_dataset/images/1478896955786775104.jpg     8   230   141   286   
35067   kaggle_dataset/images/1478896697564055470.jpg    77   325   111   285   
199495  kaggle_dataset/images/1478899311786327140.jpg     0   175    94   277   
38481   kaggle_dataset/images/1478896953501919334.jpg     1   2

In [57]:
# araba içermeyen frameleri bul

# araba içeren tüm frameleri at
frames_without_car = train_labels.loc[train_labels['class_id'] != 1]
car_frames = train_labels.loc[train_labels['class_id'] == 1]['frame']
frames_without_car = frames_without_car[~frames_without_car['frame'].isin(car_frames)]

#duplicate frameleri at
frames_without_car = frames_without_car.drop_duplicates(subset=['frame'], keep="first")

# araba sayısının 3 katı araba olmayan olmalı 
numberOfNonCarImages = numberOfCarImages * 3
n = min(numberOfNonCarImages, len(frames_without_car))
frames_without_car = frames_without_car.sample(n=n, random_state=randomSeed)

# image dizilerini değiştir
frames_without_car["frame"] = (
    "kaggle_dataset/images/" +
    frames_without_car["frame"].astype(str)
)

# eksikleri airship ile tamamla
numberOfMissingImage = numberOfNonCarImages - n
aircraftSet = pd.read_csv("kaggle_dataset_aircraft/ImageSets/Main/train.txt", header=None)

if aircraftSet.shape[0] < numberOfMissingImage:
    print("Yeterli resim yok. Araba içeren resim sayısını düşürmeyi deneyin")
    exit()

# random seç 
aircraftSet = aircraftSet.sample(numberOfMissingImage, random_state=randomSeed)
for i in range(numberOfMissingImage):
    image_id = aircraftSet.iloc[i, 0]

    frames_without_car.loc[len(frames_without_car)] = {
        "frame": f"kaggle_dataset_aircraft/JPEGImages/{image_id}.jpg",
        "xmin": 0,
        "ymin": 0,
        "xmax": 1,
        "ymax": 1,
        "class_id": 3,
        "contains_car": False
    }



if(displayImages):
    for index, row in frames_without_car.iterrows():
        image_path = row['frame']
        image = cv2.imread(image_path)
        cv2.rectangle(image, (row['xmin'], row['ymin']), (row['xmax'], row['ymax']), (0, 255, 0), 2)
        cv2.imshow("Frames wo car", image)
        print(row['frame'])
        key = cv2.waitKey(0)
        if key == 27:  # ESC tuşuna basıldığında döngüyü kır
            break
    cv2.destroyAllWindows()

# contains_car adında bool bir sütun ekle
frames_without_car['contains_car'] = False

if(printDebug):
    print(frames_without_car)
    print(frames_without_car.shape)



kaggle_dataset/images/1478900586905748308.jpg
kaggle_dataset/images/1478901293590165447.jpg
                                                frame  xmin  xmax  ymin  ymax  \
215778  kaggle_dataset/images/1478900586905748308.jpg   231   279   122   198   
221300  kaggle_dataset/images/1478901293590165447.jpg    76    83   105   128   
55267   kaggle_dataset/images/1478898276037645304.jpg    82    93   147   172   
211665  kaggle_dataset/images/1478900069887299039.jpg   161   177    86   115   
3236    kaggle_dataset/images/1478020390196268259.jpg     0    23   133   168   
...                                               ...   ...   ...   ...   ...   
1195      kaggle_dataset_aircraft/JPEGImages/2996.jpg     0     1     0     1   
1196      kaggle_dataset_aircraft/JPEGImages/1010.jpg     0     1     0     1   
1197      kaggle_dataset_aircraft/JPEGImages/1521.jpg     0     1     0     1   
1198      kaggle_dataset_aircraft/JPEGImages/1195.jpg     0     1     0     1   
1199        kaggl

In [58]:
# geçerleme ve test setini oluştur   

# araba içeren framleri 2 eşit parçaya böl. önce karıştır sonra böl
frames_only_with_car = frames_only_with_car.sample(frac=1, random_state=randomSeed)
frames_only_with_car_valid = frames_only_with_car.iloc[:frames_only_with_car.shape[0]//2]
frames_only_with_car_test = frames_only_with_car.iloc[frames_only_with_car.shape[0]//2:]

if(printDebug):
    print(frames_only_with_car_valid.shape)
    print(frames_only_with_car_test.shape)


# araba içermeyen framleri 2 eşit parçaya böl. önce karıştır sonra böl
frames_without_car = frames_without_car.sample(frac=1, random_state=randomSeed)
frames_without_car_valid = frames_without_car.iloc[:frames_without_car.shape[0]//2]
frames_without_car_test = frames_without_car.iloc[frames_without_car.shape[0]//2:]

if(printDebug):
    print(frames_without_car_valid.shape)
    print(frames_without_car_test.shape)


# valid ve test setlerini birleştir
valid_set = pd.concat([frames_only_with_car_valid, frames_without_car_valid], ignore_index=True)
test_set = pd.concat([frames_only_with_car_test, frames_without_car_test], ignore_index=True)

if(printDebug):
    print(valid_set.shape)
    print(test_set.shape)


# valid ve test setlerini karıştır
valid_set = valid_set.sample(frac=1, random_state=randomSeed).reset_index(drop=True)
test_set = test_set.sample(frac=1, random_state=randomSeed).reset_index(drop=True)

# valid ve test setlerini csv dosyalarına kaydet
valid_set.to_csv(validation_labels, index=False)
test_set.to_csv(test_labels, index=False)


(200, 7)
(200, 7)
(600, 7)
(600, 7)
(800, 7)
(800, 7)
